In [38]:
from google.cloud import bigquery
from google.oauth2 import service_account

#### Create client and link dataset Google BigQuery

In [39]:
def create_biguery_client ():
    # Set up authentication using a service account
    credentials = service_account.Credentials.from_service_account_file('bq_key.json')
    # Create a BigQuery client using the credentials
    client = bigquery.Client(credentials=credentials, project=credentials.project_id)
    print(f"The Client:\n ${client}\n was successfully created.")
    return client


In [40]:
client = create_biguery_client()

The Client:
 $<google.cloud.bigquery.client.Client object at 0x0000018BD0302320>
 was successfully created.


#### Construct a reference to a BigQuery dataset.

chicago_taxi_trips

In [41]:
project_id = "bigquery-public-data"
dataset_id = "chicago_taxi_trips" 
def create_dataset(client):
    # Construct a reference to the "stackoverflow" dataset
    dataset_ref = client.dataset(f"{dataset_id}", project=f"{project_id}")

    # API request - fetch the dataset
    dataset = client.get_dataset(dataset_ref)
    print(f"The dataset:\n ${dataset}\n has been successfully created ")
    return dataset_ref, dataset

In [42]:
dataset, dataset_ref = create_dataset(client)

The dataset:
 $Dataset(DatasetReference('bigquery-public-data', 'chicago_taxi_trips'))
 has been successfully created 


#### Explore the data.

List the available tables.

In [43]:
# Get a list of available tables
tables = list(client.list_tables(dataset))
list_of_tables = [table.table_id for table in tables]

# Print the list of tables (uncomment print type for debug)
# print(list_of_tables)
# print(type(list_of_tables))
for table in list_of_tables:
    print(table)


taxi_trips


#### Construct a reference to the relevant table
##### taxi_trips

In [44]:
# Query to show rolling average of the daily number of taxi 
# trips from January 1, 2016 to March 31, 2016.
table_id = "taxi_trips"
avg_num_trips_query = f"""
    WITH trips_by_day AS
    (
    SELECT DATE(trip_start_timestamp) AS trip_date, COUNT(*) AS num_trips
    FROM `{project_id}.{dataset_id}.{table_id}`
    WHERE trip_start_timestamp >= '2016-01-01' AND trip_start_timestamp < '2016-04-01'
    GROUP BY trip_date
    ORDER BY trip_date
    )
    SELECT trip_date, 
        AVG(num_trips) 
        OVER (
            ORDER BY trip_date ROWS 
            BETWEEN 3 PRECEDING AND 3
            FOLLOWING) AS rolling_avg_trips
    FROM trips_by_day
                      """

# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(avg_num_trips_query, job_config=dry_run_config)
    
# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

This query will process 1693243672 bytes.


Run the query with safe_config.

In [45]:
import warnings
# Suppress bigquery storage module warning
# Could also install 'pip install google-cloud-bigquery-storage'
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")

# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(avg_num_trips_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
results = my_query_job.to_dataframe()

# print(results)
results.head()


,trip_date,rolling_avg_trips
0,2016-02-27,92965.142857
1,2016-01-14,86916.428571
2,2016-03-18,95345.142857
3,2016-01-25,82466.142857
4,2016-01-28,80813.142857


In [46]:
# Query to separate and order trips by pickup community area and trip start timestamp, and assign a trip number to each trip for a given pickup community area on October 3, 2013.
trip_number_query = f"""
                    SELECT pickup_community_area,
                        trip_start_timestamp,
                        trip_end_timestamp,
                    RANK() OVER (PARTITION BY pickup_community_area 
                              ORDER BY trip_start_timestamp) AS trip_number
                    FROM `{project_id}.{dataset_id}.{table_id}`
                    WHERE DATE(trip_start_timestamp) = '2013-10-03'
                    """
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(trip_number_query, job_config=dry_run_config)
    
# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

This query will process 4886175784 bytes.


Run the query with safe_config.

In [47]:
# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(trip_number_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
first_results = my_query_job.to_dataframe()
first_results.head()


,pickup_community_area,trip_start_timestamp,trip_end_timestamp,trip_number
0,59,2013-10-03 02:00:00+00:00,2013-10-03 02:00:00+00:00,1
1,59,2013-10-03 03:00:00+00:00,2013-10-03 03:15:00+00:00,2
2,59,2013-10-03 09:15:00+00:00,2013-10-03 09:45:00+00:00,3
3,59,2013-10-03 09:15:00+00:00,2013-10-03 09:15:00+00:00,3
4,59,2013-10-03 09:30:00+00:00,2013-10-03 09:30:00+00:00,5


In [48]:
break_time_query = f"""
    SELECT taxi_id,
        trip_start_timestamp,
        trip_end_timestamp,
        TIMESTAMP_DIFF(
            trip_start_timestamp, 
            LAG(trip_end_timestamp, 1)
            OVER (
            PARTITION BY taxi_id 
            ORDER BY trip_start_timestamp),
        MINUTE) AS prev_break
        FROM `{project_id}.{dataset_id}.{table_id}`
                    WHERE DATE(trip_start_timestamp) = '2013-10-03'
"""
# Create a "QueryJobConfig" object to set the "dry_run"
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

# API request - dry run query to estimate cost of query
dry_run_query_job = client.query(break_time_query, job_config=dry_run_config)
    
# Print estimated number of bytes processed by the query
print(f"This query will process {dry_run_query_job.total_bytes_processed} bytes.")

This query will process 30901549262 bytes.


Run the query with safe_config

In [ ]:
import warnings
# Suppress bigquery storage module warning
# Could also install 'pip install google-cloud-bigquery-storage'
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")

# Set up the query (a real service would have good error handling for 
# queries that scan too much data)
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10)      
my_query_job = client.query(break_time_query, job_config=safe_config)

# API request - run the query, and return a pandas DataFrame
results = my_query_job.to_dataframe()

# print(results)
results.head()
